# Test Notebook - Part 2
### Test các hàm sau:
- cholesky_custom
- diagonalize_matrix
- verify_diagonalization
- matrix_power_via_diagonalization

Moi ham co 5 test case.

In [1]:
import os
import sys
import unittest

import numpy as np

THIS_DIR = os.getcwd()
if THIS_DIR not in sys.path:
    sys.path.append(THIS_DIR)

from decomposition import cholesky_custom
from diagonalization import (
    diagonalize_matrix,
    verify_diagonalization,
    matrix_power_via_diagonalization,
)

def naive_power(A, k):
    n = len(A)
    out = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    for _ in range(k):
        nxt = [[0.0 for _ in range(n)] for _ in range(n)]
        for i in range(n):
            for t in range(n):
                for j in range(n):
                    nxt[i][j] += out[i][t] * A[t][j]
        out = nxt
    return out

In [2]:
class TestCholeskyCustomPart2(unittest.TestCase):
    def test_case_1_spd_3x3(self):
        A = [[4.0, 12.0, -16.0], [12.0, 37.0, -43.0], [-16.0, -43.0, 98.0]]
        L = np.array(cholesky_custom(A), dtype=float)
        self.assertTrue(np.allclose(L @ L.T, np.array(A, dtype=float), atol=1e-9))

    def test_case_2_spd_diagonal(self):
        A = [[9.0, 0.0, 0.0], [0.0, 16.0, 0.0], [0.0, 0.0, 25.0]]
        L = np.array(cholesky_custom(A), dtype=float)
        self.assertTrue(np.allclose(np.diag(L), [3.0, 4.0, 5.0], atol=1e-9))

    def test_case_3_spd_4x4(self):
        A = [[25.0, 15.0, -5.0, 10.0], [15.0, 18.0, 0.0, 6.0], [-5.0, 0.0, 11.0, 2.0], [10.0, 6.0, 2.0, 29.0]]
        L = np.array(cholesky_custom(A), dtype=float)
        self.assertTrue(np.allclose(L @ L.T, np.array(A, dtype=float), atol=1e-8))

    def test_case_4_non_symmetric_raises(self):
        A = [[4.0, 1.0], [0.0, 3.0]]
        with self.assertRaises(ValueError):
            cholesky_custom(A)

    def test_case_5_not_positive_definite_raises(self):
        A = [[1.0, 2.0], [2.0, 1.0]]
        with self.assertRaises(ValueError):
            cholesky_custom(A)

In [3]:
class TestDiagonalizationPart2(unittest.TestCase):
    def test_case_1_diagonal_matrix(self):
        A = [[5.0, 0.0], [0.0, 2.0]]
        P, D, P_inv = diagonalize_matrix(A)
        ok, err = verify_diagonalization(A, P, D, P_inv)
        self.assertTrue(ok)
        self.assertLess(err, 1e-8)

    def test_case_2_regular_2x2(self):
        A = [[4.0, 1.0], [2.0, 3.0]]
        P, D, P_inv = diagonalize_matrix(A)
        ok, _ = verify_diagonalization(A, P, D, P_inv)
        self.assertTrue(ok)

    def test_case_3_jordan_not_diagonalizable(self):
        A = [[1.0, 1.0], [0.0, 1.0]]
        with self.assertRaises(ValueError):
            diagonalize_matrix(A)

    def test_case_4_non_square_raises(self):
        A = [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]
        with self.assertRaises(ValueError):
            diagonalize_matrix(A)

    def test_case_5_too_strict_condition_threshold(self):
        A = [[2.0, 0.0], [0.0, 3.0]]
        with self.assertRaises(ValueError):
            diagonalize_matrix(A, cond_threshold=0.5)

In [4]:
class TestVerifyDiagonalizationPart2(unittest.TestCase):
    def test_case_1_true_for_valid_decomposition(self):
        A = [[4.0, 1.0], [2.0, 3.0]]
        P, D, P_inv = diagonalize_matrix(A)
        ok, err = verify_diagonalization(A, P, D, P_inv)
        self.assertTrue(ok)
        self.assertLess(err, 1e-6)

    def test_case_2_false_for_wrong_inverse(self):
        A = [[4.0, 1.0], [2.0, 3.0]]
        P, D, P_inv = diagonalize_matrix(A)
        P_inv_bad = [row[:] for row in P_inv]
        P_inv_bad[0][0] += 0.1
        ok, err = verify_diagonalization(A, P, D, P_inv_bad)
        self.assertFalse(ok)
        self.assertGreater(err, 1e-6)

    def test_case_3_true_when_tolerance_relaxed(self):
        A = [[4.0, 1.0], [2.0, 3.0]]
        P, D, P_inv = diagonalize_matrix(A)
        P_inv_bad = [row[:] for row in P_inv]
        P_inv_bad[0][0] += 1e-6
        ok, _ = verify_diagonalization(A, P, D, P_inv_bad, atol=1e-4, rtol=1e-4)
        self.assertTrue(ok)

    def test_case_4_non_square_A_raises(self):
        with self.assertRaises(ValueError):
            verify_diagonalization([[1.0, 2.0, 3.0]], [[1.0]], [[1.0]], [[1.0]])

    def test_case_5_error_is_numeric(self):
        A = [[2.0, 0.0], [0.0, 3.0]]
        P, D, P_inv = diagonalize_matrix(A)
        ok, err = verify_diagonalization(A, P, D, P_inv)
        self.assertIsInstance(err, float)
        self.assertTrue(ok)

In [5]:
class TestMatrixPowerPart2(unittest.TestCase):
    def test_case_1_power_zero_identity(self):
        A = [[4.0, 1.0], [2.0, 3.0]]
        out = matrix_power_via_diagonalization(A, 0)
        self.assertTrue(np.allclose(out, [[1.0, 0.0], [0.0, 1.0]], atol=1e-9))

    def test_case_2_power_one_equals_A(self):
        A = [[4.0, 1.0], [2.0, 3.0]]
        out = matrix_power_via_diagonalization(A, 1)
        self.assertTrue(np.allclose(out, A, atol=1e-8))

    def test_case_3_power_three_vs_naive(self):
        A = [[4.0, 1.0], [2.0, 3.0]]
        out = matrix_power_via_diagonalization(A, 3)
        expect = naive_power(A, 3)
        self.assertTrue(np.allclose(out, expect, atol=1e-6))

    def test_case_4_negative_power_raises(self):
        with self.assertRaises(ValueError):
            matrix_power_via_diagonalization([[4.0, 1.0], [2.0, 3.0]], -1)

    def test_case_5_not_diagonalizable_raises(self):
        with self.assertRaises(ValueError):
            matrix_power_via_diagonalization([[1.0, 1.0], [0.0, 1.0]], 2)

In [6]:
suite = unittest.TestSuite()
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestCholeskyCustomPart2))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestDiagonalizationPart2))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestVerifyDiagonalizationPart2))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestMatrixPowerPart2))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print('--- SUMMARY ---')
print('run =', result.testsRun)
print('failures =', len(result.failures))
print('errors =', len(result.errors))

test_case_1_spd_3x3 (__main__.TestCholeskyCustomPart2.test_case_1_spd_3x3) ... ok
test_case_2_spd_diagonal (__main__.TestCholeskyCustomPart2.test_case_2_spd_diagonal) ... ok
test_case_3_spd_4x4 (__main__.TestCholeskyCustomPart2.test_case_3_spd_4x4) ... ok
test_case_4_non_symmetric_raises (__main__.TestCholeskyCustomPart2.test_case_4_non_symmetric_raises) ... ok
test_case_5_not_positive_definite_raises (__main__.TestCholeskyCustomPart2.test_case_5_not_positive_definite_raises) ... ok
test_case_1_diagonal_matrix (__main__.TestDiagonalizationPart2.test_case_1_diagonal_matrix) ... ok
test_case_2_regular_2x2 (__main__.TestDiagonalizationPart2.test_case_2_regular_2x2) ... ok
test_case_3_jordan_not_diagonalizable (__main__.TestDiagonalizationPart2.test_case_3_jordan_not_diagonalizable) ... ok
test_case_4_non_square_raises (__main__.TestDiagonalizationPart2.test_case_4_non_square_raises) ... ok
test_case_5_too_strict_condition_threshold (__main__.TestDiagonalizationPart2.test_case_5_too_strict

--- SUMMARY ---
run = 20
failures = 0
errors = 0
